# Lista Interpolação — Interpolação Polinomial

In [1]:
import numpy as np
import matplotlib.pyplot as plt

def lagrange(xs, ys, xq):
    xs, ys = np.asarray(xs,float), np.asarray(ys,float)
    result = 0.0
    for i in range(len(xs)):
        L = 1.0
        for j in range(len(xs)):
            if j != i: L *= (xq-xs[j])/(xs[i]-xs[j])
        result += ys[i]*L
    return result

def newton_interp(xs, ys, xq):
    xs, ys = np.asarray(xs,float), np.asarray(ys,float)
    n = len(xs)
    c = ys.copy()
    for j in range(1, n):
        c[j:] = (c[j:]-c[j-1:-1 or None]) / (xs[j:]-xs[:n-j])
    result = c[-1]
    for k in range(n-2,-1,-1): result = result*(xq-xs[k]) + c[k]
    return result


## Questão 1
Diodo I-V — interpolação de Lagrange V(I) com 5 nós.

In [2]:
Is_d, nu, VT = 10.0, 1.8, 26e-3
V_nos = np.linspace(0.1, 0.7, 5)
I_nos = Is_d*(np.exp(V_nos/(nu*VT)) - 1)

V_mid = 0.5*(V_nos[:-1]+V_nos[1:])
I_mid = Is_d*(np.exp(V_mid/(nu*VT)) - 1)
V_int = np.array([lagrange(I_nos, V_nos, i) for i in I_mid])

print(f"{'V_exato':>10} {'V_interp':>10} {'erro':>10}")
for ve, vi in zip(V_mid, V_int):
    print(f"{ve:>10.6f} {vi:>10.6f} {vi-ve:>10.6f}")


   V_exato   V_interp       erro
  0.175000   0.125959  -0.049041
  0.325000   0.746992   0.421992
  0.475000 -56.328369 -56.803369
  0.625000 174096.325959 174095.700959


## Questão 2
Termistor T(R) — interpolação polinomial com 7 nós.

In [3]:
R0, T0, beta = 10000.0, 298.0, 4000.0
Tc_nos = np.linspace(20, 100, 7); Tk_nos = Tc_nos+273.15
R_nos = R0*np.exp(beta*(1/Tk_nos - 1/T0))

Tc_mid = 0.5*(Tc_nos[:-1]+Tc_nos[1:])
Tk_mid = Tc_mid+273.15; R_mid = R0*np.exp(beta*(1/Tk_mid - 1/T0))
Tc_int = np.array([newton_interp(R_nos, Tc_nos, r) for r in R_mid])

print(f"{'T_exata':>9} {'T_interp':>10} {'erro':>10}")
for te, ti in zip(Tc_mid, Tc_int):
    print(f"{te:>9.4f} {ti:>10.4f} {ti-te:>10.4f}")


  T_exata   T_interp       erro
  26.6667  -214.0892  -240.7558
  40.0000    48.4462     8.4462
  53.3333    52.5194    -0.8140
  66.6667    66.8475     0.1808
  80.0000    79.9114    -0.0886
  93.3333    93.4378     0.1044


## Questão 3
Carga elétrica às 4h30 — spline linear e polinômio global.

In [4]:
hora = np.array([1,2,3,4,5,6,7,8], dtype=float)
MW   = np.array([45,42,40,38,36,40,50,65], dtype=float)
xq   = 4.5

# Spline linear (interpolação entre h=4 e h=5)
i = np.searchsorted(hora, xq) - 1
spline = MW[i] + (MW[i+1]-MW[i])/(hora[i+1]-hora[i]) * (xq-hora[i])

# Polinômio de Newton global
poly = newton_interp(hora, MW, xq)

print(f"Spline linear em 4.5h = {spline:.4f} MW")
print(f"Polinômio global em 4.5h = {poly:.4f} MW")


Spline linear em 4.5h = 37.0000 MW
Polinômio global em 4.5h = 36.6250 MW


## Questão 4
f(4.8) com 3 nós (2, 5, 7) — Newton e Lagrange.

In [5]:
xs_loc = np.array([2.0, 5.0, 7.0])
ys_loc = np.array([2.0, 4.0, 8.0])
xq = 4.8

print(f"Newton  f(4.8) = {newton_interp(xs_loc, ys_loc, xq):.6f}")
print(f"Lagrange f(4.8) = {lagrange(xs_loc, ys_loc, xq):.6f}")


Newton  f(4.8) = 3.717333
Lagrange f(4.8) = 3.717333


## Questão 5
f(2.8) com diferenças divididas de Newton — graus 1, 2 e 3.

In [6]:
xs_all = np.array([1.6, 2.0, 2.5, 3.2, 4.0, 4.5])
ys_all = np.array([2.0, 8.0, 14.0, 15.0, 8.0, 2.0])
xq = 2.8

nos = {1:[2.5,3.2], 2:[2.0,2.5,3.2], 3:[2.0,2.5,3.2,4.0]}
for g, idx in nos.items():
    xi = np.array(idx); yi = np.array([ys_all[np.where(xs_all==v)[0][0]] for v in xi])
    print(f"Grau {g}: f(2.8) = {newton_interp(xi, yi, xq):.6f}")


Grau 1: f(2.8) = 14.428571
Grau 2: f(2.8) = 15.485714
Grau 3: f(2.8) = 15.388571


## Questão 6
Interpolação inversa — encontrar x tal que f(x) = 0.85.

In [7]:
xs = np.array([0,1,2,3,4,5], dtype=float)
ys = np.array([0,0.5,0.8,0.9,0.941176,0.961538], dtype=float)
yq = 0.85; x_exato = (1.7/0.15)**(1/3)

# inverter: interpolar x em função de y
xi = ys[[1,2,3,4]]; yi = xs[[1,2,3,4]]
x_int = newton_interp(xi, yi, yq)
print(f"x exato    = {x_exato:.8f}")
print(f"x interp   = {x_int:.8f}")
print(f"erro abs   = {abs(x_int-x_exato):.2e}")


x exato    = 2.24622137
x interp   = 2.29068970
erro abs   = 4.45e-02


## Questão 7
Circuito com varistor — interpolação I(E₂) e cálculo de E.

In [8]:
E2 = np.array([0,0.5,1,1.5,2,2.5,3,3.5,4,4.5], dtype=float)
I  = np.array([0,0.0125,0.06,0.195,0.5,1.0875,2.1,3.71,6.12,9.5625], dtype=float)
E2q, R1 = 2.3, 10.0
I_q = lagrange(E2, I, E2q)
E1  = R1*I_q
print(f"I(2.3 V) = {I_q:.6f} A")
print(f"E1 = {E1:.6f} V")
print(f"E  = {E1+E2q:.6f} V")


I(2.3 V) = 0.810152 A
E1 = 8.101520 V
E  = 10.401520 V
